# Inference on the Kaggle CIFAR-10 Test Set

Loads a trained model (weights saved by `notebooks/02_model.ipynb` as `model.pt`) and produces class predictions for every image in the Kaggle CIFAR-10 test set, saving them as a `id,label` submission CSV.

In [4]:
import os
import json
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.dataset import CifarTestDataset
from src.model import SimpleCNNConv3
from src.predict import CIFAR10_CLASSES, load_data_stats, build_predict_transform
from src.training import device

## Configuration

`EXPERIMENT_DIR` defaults to whatever `configs/serving_config.json` (`model_experiment`) points at — the same single source of truth used by `src/api.py`, the `Dockerfile`, and the `Makefile`. Override the cell below if you want to run inference against a different experiment without touching that config.

In [ ]:
# Directory of the Kaggle CIFAR-10 test images.
TEST_DIR = Path("/Users/vyankov/.cache/kagglehub/competitions/cifar-10/test")

# Experiment whose saved model (state_dict) should be used for inference.
with open("../configs/serving_config.json", encoding="utf-8") as file:
    serving_config = json.load(file)
EXPERIMENT_DIR = Path(f"../experiments/{serving_config['model_experiment']}")
MODEL_PATH = EXPERIMENT_DIR / "model.pt"
MODEL_CONFIG_PATH = EXPERIMENT_DIR / "config.json"

# Where to write the resulting predictions.
OUTPUT_CSV = Path("../data/processed/submission.csv")

BATCH_SIZE = 256
NUM_WORKERS = 4

## Load the trained model

In [ ]:
with open(MODEL_CONFIG_PATH, encoding="utf-8") as file:
    model_config = json.load(file)

dropout = model_config["model"]["dropout"]

# Prefer the class list train.py persists alongside the checkpoint (same
# fallback logic as src/api.py), so an older experiment without classes.json
# still works.
classes_path = EXPERIMENT_DIR / "classes.json"
if classes_path.exists():
    with open(classes_path, encoding="utf-8") as file:
        class_names = json.load(file)
else:
    class_names = CIFAR10_CLASSES

model = SimpleCNNConv3(num_classes=len(class_names), dropout=dropout)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

print(f"Loaded weights from {MODEL_PATH} (dropout={dropout})")

## Build the preprocessing transform

Uses the same normalization stats (`configs/data_stats.json`) applied during training/validation.

In [8]:
mean, std = load_data_stats()
transform = build_predict_transform(mean, std)

## Test dataset

`CifarTestDataset` (see `src/dataset.py`) reads every image in `TEST_DIR`, keeping track of its numeric `id` (from the file name) so predictions can be written out against the right id.

In [9]:
test_dataset = CifarTestDataset(TEST_DIR, transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(f"Found {len(test_dataset)} test images.")

Found 300000 test images.


## Run inference

In [ ]:
ids, predicted_labels = [], []

with torch.no_grad():
    for images, image_ids in tqdm(test_loader, desc="Predicting"):
        images = images.to(device)
        outputs = model(images)
        predicted_indices = outputs.argmax(dim=1).cpu().tolist()

        ids.extend(image_ids.tolist())
        predicted_labels.extend(class_names[index] for index in predicted_indices)

## Save predictions to CSV

In [11]:
predictions_df = pd.DataFrame({"id": ids, "label": predicted_labels}).sort_values("id")

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
predictions_df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(predictions_df)} predictions to {OUTPUT_CSV}")
predictions_df.head()

Saved 300000 predictions to ../data/processed/submission.csv


,id,label
0,1,deer
1,2,airplane
2,3,automobile
3,4,ship
4,5,airplane
